# Trying to batch regions for faster inference

In [1]:
from zettelsortierung.OCR import PaddleOCR
from zettelsortierung.Zettelwerk import Zettelsammlung
from zettelsortierung.RegionDetection import CV2RegionDetector
from zettelsortierung.ImageAnnotation import BoundingBox, AnnotatedImage
from zettelsortierung.OCRPreprocessing import PaddleOCRPreprocessor
from typing import Iterable
import os
import cv2
import numpy as np


def sort_boundingbox_data(region_data: dict) -> dict:
    # Sorts Patches by Width
    return dict(sorted(region_data.items(), key=lambda item: item[0][1][2]))

def batch_generator(patch_data: dict, batch_size: int) -> Iterable[tuple[list]]:
    image_paths_batch = []
    im_patches_batch = []
    for counter, (image_path_and_boundingbox, im_patch) in enumerate(patch_data.items()):
        image_paths_batch.append(image_path_and_boundingbox)
        im_patches_batch.append(im_patch)
        if (counter+1) % batch_size == 0:
            yield image_paths_batch, im_patches_batch
            image_paths_batch = []
            im_patches_batch = []

def pad_and_stack_patches(im_patches_batch: list[np.ndarray]) -> np.ndarray:
    w_max = max(im_patch.shape[1] for im_patch in im_patches_batch)
    padded_im_patches_batch = []
    for im_patch in im_patches_batch:
        h, w, c = im_patch.shape
        padded_im_region = np.pad(im_patch, ((0, 0), (0, w_max-w), (0, 0)), mode='constant', constant_values=0)
        padded_im_patches_batch.append(padded_im_region)
    return np.stack(padded_im_patches_batch)



root = os.getenv('ZETTELSAMMLUNG_ROOT')
sammlung = Zettelsammlung.collect_zettel(root, 1000)#[:2]
path_to_annotatedimage = {zettel.recto_file_path: AnnotatedImage(zettel.recto_file_path) for zettel in sammlung}

model = PaddleOCR()
preprocessor = PaddleOCRPreprocessor()
region_detector = CV2RegionDetector()

#patch_data = {}

21it [00:07,  2.68it/s]


In [2]:
'''

for zettel in sammlung:
    image_path = zettel.recto_file_path
    full_image = cv2.imread(image_path)
    y_max, x_max, c_max = full_image.shape

    unpadded_boundingboxes = region_detector.detect_regions(image=full_image)
    boundingboxes = preprocessor.pad_boundingboxes(unpadded_boundingboxes, x_max, y_max, padding=10)

    unresized_im_patches = preprocessor.cut_patches(full_image, boundingboxes)
    unconverted_im_patches = preprocessor.resize_im_patches(unresized_im_patches, h_new=48)
    im_patches = preprocessor.convert_im_patches_to_RGB(unconverted_im_patches)

    for boundingbox, im_patch in zip(boundingboxes, im_patches):
        patch_data[(image_path, boundingbox)] = im_patch
'''

'\n\nfor zettel in sammlung:\n    image_path = zettel.recto_file_path\n    full_image = cv2.imread(image_path)\n    y_max, x_max, c_max = full_image.shape\n\n    unpadded_boundingboxes = region_detector.detect_regions(image=full_image)\n    boundingboxes = preprocessor.pad_boundingboxes(unpadded_boundingboxes, x_max, y_max, padding=10)\n\n    unresized_im_patches = preprocessor.cut_patches(full_image, boundingboxes)\n    unconverted_im_patches = preprocessor.resize_im_patches(unresized_im_patches, h_new=48)\n    im_patches = preprocessor.convert_im_patches_to_RGB(unconverted_im_patches)\n\n    for boundingbox, im_patch in zip(boundingboxes, im_patches):\n        patch_data[(image_path, boundingbox)] = im_patch\n'

In [3]:
from multiprocessing import Pool, cpu_count

def inner(zettel):
    im_path = zettel.recto_file_path
    full_image = cv2.imread(im_path)
    y_max, x_max, c_max = full_image.shape

    unpadded_boundingboxes = region_detector.detect_regions(image=full_image)
    patch_dict = {}

    for unpadded_boundingbox in unpadded_boundingboxes:
        # Pad boundingbox
        padding = 10
        x_old, y_old, w_old, h_old = unpadded_boundingbox
        x = max(0, x_old-padding)
        y = max(0, y_old-padding)
        w = min(w_old+2*padding, x_max-x)
        h = min(h_old+2*padding, y_max-y)
        boundingbox = BoundingBox(x, y, w, h)

        # Cout out patche
        unresized_im_patch = full_image[y:y+h, x:x+w]
        # Resize patch
        h_new = 48
        w_new = int(w * h_new / h)
        unconverted_im_patch = cv2.resize(unresized_im_patch, (w_new, h_new))#, interpolation=cv2.INTER_AREA)
        # Convert BGR to RGB
        im_patch = cv2.cvtColor(unconverted_im_patch, cv2.COLOR_BGR2RGB)
        
        # Store result
        patch_dict[(im_path, boundingbox)] = im_patch

    return patch_dict


#results = [inner(zettel) for zettel in sammlung]
with Pool(12) as pool:
    results = list(pool.imap_unordered(inner, sammlung))

In [4]:
from functools import reduce
import operator
path_bb_to_patch = reduce(operator.ior, results, {})

In [5]:
'''
sorted_patch_data = sort_boundingbox_data(patch_data)
patch_labels = []
for paths_boundingboxes_list, im_patchs_list in batch_generator(sorted_patch_data, batch_size=4):
    batched_im_patchs = pad_and_stack_patches(im_patchs_list)
    labels = model(batched_im_patchs)
    patch_labels.extend(labels)
'''

'\nsorted_patch_data = sort_boundingbox_data(patch_data)\npatch_labels = []\nfor paths_boundingboxes_list, im_patchs_list in batch_generator(sorted_patch_data, batch_size=4):\n    batched_im_patchs = pad_and_stack_patches(im_patchs_list)\n    labels = model(batched_im_patchs)\n    patch_labels.extend(labels)\n'

In [10]:
def labeling_inner(lists):
    paths_bbs_list, im_patches_list = lists
    batched_im_patches = pad_and_stack_patches(im_patches_list)
    labels = model(batched_im_patches)
    return {path_boundingbox: label for path_boundingbox, label in zip(paths_bbs_list, labels)}


sorted_path_bb_to_patch = sort_boundingbox_data(path_bb_to_patch)
results = [labeling_inner(lists) for lists in batch_generator(sorted_path_bb_to_patch, batch_size=4)]
path_bb_to_label = reduce(operator.ior, results, {})

In [7]:
for (image_path, boundingbox), region_label in path_bb_to_label.items():
    path_to_annotatedimage[image_path].add_text_region(boundingbox, region_label)

In [8]:
for annotated_image in path_to_annotatedimage.values():
    annotated_image.display_image()
    break